# Σ-Model Paper 02 — Phase 06: Tiered Cross-Benchmark & Architecture Scaling Sweeps

### Authorized Experimental Matrix Specification (Tiered Design):
- **Tier 1 — Primary Change-Point Falsification (720 Runs):**
  - 4 Benchmarks: H-Bar, SCAN (`add_primitive_jump`), COGS/ReCOGS, PCFG-SET
  - 6 λ Grid Levels: 0.000 (Subcritical), 0.015 (Boundary Approach), 0.020 (Boundary Entry), 0.025 (Locked Critical Estimate), 0.030 (Boundary Exit), 0.500 (Supercritical Reference)
  - Architecture: Transformer 2-Layer Baseline (d_model=128, n_heads=4, n_layers=2)
  - Sample Size: n = 30 seeds/cell (s_i = run_id * 42 + 7) ==> 4 x 1 x 6 x 30 = 720 runs
- **Tier 2 — Exploratory Architecture Universality (240 Runs):**
  - 2 Scaling Architectures: Transformer 4-Layer Scaled (d_model=256), GRU Recurrent Seq2Seq Baseline
  - 3 λ Grid Levels: 0.000 (Subcritical), 0.025 (Critical Boundary), 0.500 (Supercritical)
  - 4 Benchmarks x 2 Architectures x 3 λ Levels x 10 seeds = 240 runs
- **Total Planned Runs:** 720 (Tier 1 Primary) + 240 (Tier 2 Exploratory) = 960 runs (~16.7 GPU hours)
- **Deep Diagnostic Probes (In Situ at Every 50 Steps):**
  - Whitened GCA ($g_A^{\text{proj}}$)
  - Linear & RBF CKA ($RGA$)
  - Lanczos Hessian top eigenvalue $\lambda_{\max}(H)$
  - Attention Head Schema Specialization
- **Compute Target:** Tesla T4 / P100 GPU (with PyTorch AMP)
- **Estimated Runtime:** $\sim 16.7\text{ hours}$ (well within the 20 GPU-hour cap)


In [ ]:
# ===========================================================================
# 0. System Setup & GPU Verification
# ===========================================================================
import os
import sys
import time
import math
import random
import zipfile
import pickle
import hashlib
from dataclasses import dataclass, field, asdict
from pathlib import Path

import numpy as np
import scipy.linalg
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast_mode, grad_scaler
from torch.utils.data import Dataset, DataLoader

print('='*75)
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device Name : {torch.cuda.get_device_name(0)}')
    print(f'Device Memory   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

OUT_DIR = Path('/kaggle/working/p06_output')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output Directory: {OUT_DIR}')
print('='*75)


In [ ]:
# ===========================================================================
# 1. Benchmark Synthesizers & Grammars (H-Bar, SCAN, COGS, PCFG-SET)
# ===========================================================================

def build_vocab():
    tokens = [
        '<pad>', '<bos>', '<eos>', '<unk>',
        # Verbs & Actions
        'walk', 'look', 'run', 'jump', 'turn', 'push', 'pull', 'lift',
        # Modifiers & Directions
        'left', 'right', 'twice', 'thrice', 'opposite', 'around',
        # Combinators & Structure
        'and', 'after', 'while', 'before', 'then', 'so',
        # Semantic Entities (COGS / PCFG)
        'agent', 'theme', 'recipient', 'goal', 'source',
        'cat', 'dog', 'ball', 'box', 'table', 'boy', 'girl',
        # Output primitives
        'I_WALK', 'I_LOOK', 'I_RUN', 'I_JUMP', 'I_TURN_LEFT', 'I_TURN_RIGHT',
        'I_PUSH', 'I_PULL', 'I_LIFT', 'I_AND', 'I_AFTER'
    ]
    vocab2idx = {tok: i for i, tok in enumerate(tokens)}
    idx2vocab = {i: tok for i, tok in enumerate(tokens)}
    return vocab2idx, idx2vocab

VOCAB2IDX, IDX2VOCAB = build_vocab()
VOCAB_SIZE = len(VOCAB2IDX)
print(f'Global Vocabulary Initialized! Size = {VOCAB_SIZE}')

class Seq2SeqDataset(Dataset):
    def __init__(self, pairs, vocab2idx):
        self.data = []
        for inp_tokens, out_tokens in pairs:
            inp_ids = [vocab2idx.get(t, vocab2idx['<unk>']) for t in inp_tokens] + [vocab2idx['<eos>']]
            out_ids = [vocab2idx.get(t, vocab2idx['<unk>']) for t in out_tokens] + [vocab2idx['<eos>']]
            self.data.append((torch.tensor(inp_ids, dtype=torch.long), torch.tensor(out_ids, dtype=torch.long)))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def pad_collate(batch):
    inps, outs = zip(*batch)
    max_inp = max(len(x) for x in inps)
    max_out = max(len(y) for y in outs)
    pad_inps = torch.zeros(len(inps), max_inp, dtype=torch.long)
    pad_outs = torch.zeros(len(outs), max_out, dtype=torch.long)
    for i, (x, y) in enumerate(zip(inps, outs)):
        pad_inps[i, :len(x)] = x
        pad_outs[i, :len(y)] = y
    return pad_inps, pad_outs

def generate_benchmark_splits(benchmark_name, seed=42):
    random.seed(seed)
    np.random.seed(seed)
    actions = ['walk', 'look', 'run', 'turn', 'push']
    mods = ['left', 'right', 'twice', 'thrice', 'opposite', 'around']
    combs = ['and', 'after']
    
    train_pairs, val_pairs, ood_pairs = [], [], []
    
    if benchmark_name == 'scan_jump':
        # Primitive jump zero-shot extrapolation
        for a in actions:
            for m in mods:
                inp = [a, m]
                out = [f'I_{a.upper()}', f'I_{m.upper()}']
                train_pairs.extend([(inp, out)] * 15)
        train_pairs.append((['jump'], ['I_JUMP']))
        for m in mods:
            for c in combs:
                for a in actions:
                    ood_pairs.append((['jump', m, c, a], ['I_JUMP', f'I_{m.upper()}', f'I_{c.upper()}', f'I_{a.upper()}']))
    elif benchmark_name == 'scan_length':
        # Length split: train <= 3 tokens, test >= 4 tokens
        for a in actions + ['jump']:
            for m in mods:
                train_pairs.extend([([a, m], [f'I_{a.upper()}', f'I_{m.upper()}'])] * 10)
                for c in combs:
                    ood_pairs.append(([a, m, c, a, m], [f'I_{a.upper()}', f'I_{m.upper()}', f'I_{c.upper()}', f'I_{a.upper()}', f'I_{m.upper()}']))
    elif benchmark_name == 'cogs':
        # Structural recursion & role inversion
        nouns = ['cat', 'dog', 'ball', 'boy', 'girl']
        verbs = ['push', 'pull', 'lift']
        for n1 in nouns:
            for v in verbs:
                for n2 in nouns:
                    if n1 != n2:
                        train_pairs.append(([n1, v, n2], [f'I_{v.upper()}', 'agent', n1, 'theme', n2]))
        for n in nouns:
            ood_pairs.append(([n, 'walk', 'while', n, 'jump'], ['I_WALK', 'agent', n, 'while', 'I_JUMP', 'agent', n]))
    else: # pcfg_set / hbar
        for a in actions + ['jump']:
            for m in mods:
                train_pairs.extend([([a, m], [f'I_{a.upper()}', f'I_{m.upper()}'])] * 8)
                for c in combs:
                    ood_pairs.append(([a, m, c, a], [f'I_{a.upper()}', f'I_{m.upper()}', f'I_{c.upper()}', f'I_{a.upper()}']))
    
    random.shuffle(train_pairs)
    val_pairs = train_pairs[:len(train_pairs)//10]
    train_pairs = train_pairs[len(train_pairs)//10:]
    return train_pairs, val_pairs, ood_pairs


In [ ]:
# ===========================================================================
# 2. Neural Architecture Definitions (2L-Transformer, 4L-Transformer, GRU)
# ===========================================================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=128):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class TransformerSeq2Seq(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2, dim_ff=512, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_layers, num_decoder_layers=num_layers,
            dim_feedforward=dim_ff, dropout=dropout, batch_first=True
        )
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.d_model = d_model

    def forward(self, src, tgt):
        tgt_seq_len = tgt.size(1)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_seq_len, device=src.device)
        src_emb = self.pos_encoder(self.embedding(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_encoder(self.embedding(tgt) * math.sqrt(self.d_model))
        out = self.transformer(src_emb, tgt_emb, tgt_mask=tgt_mask)
        return self.fc_out(out)

class GRUSeq2Seq(nn.Module):
    def __init__(self, vocab_size, d_model=128, num_layers=2, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.encoder = nn.GRU(d_model, d_model, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.decoder = nn.GRU(d_model, d_model, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.d_model = d_model

    def forward(self, src, tgt):
        src_emb = self.embedding(src)
        _, h_n = self.encoder(src_emb)
        tgt_emb = self.embedding(tgt)
        dec_out, _ = self.decoder(tgt_emb, h_n)
        return self.fc_out(dec_out)

def instantiate_model(arch_name, vocab_size):
    if arch_name == 'transformer_2l':
        return TransformerSeq2Seq(vocab_size, d_model=128, nhead=4, num_layers=2, dim_ff=512)
    elif arch_name == 'transformer_4l_scaled':
        return TransformerSeq2Seq(vocab_size, d_model=256, nhead=8, num_layers=4, dim_ff=1024)
    elif arch_name == 'gru_baseline':
        return GRUSeq2Seq(vocab_size, d_model=128, num_layers=2)
    raise ValueError(f'Unknown architecture: {arch_name}')


In [ ]:
# ===========================================================================
# 3. Deep Diagnostics: Whitened GCA, Lanczos Hessian, and CKA Probing
# ===========================================================================

def compute_whitened_gca(model, loss_train, loss_comp):
    params = [p for p in model.parameters() if p.requires_grad]
    grads_train = torch.autograd.grad(loss_train, params, retain_graph=True, allow_unused=True)
    grads_comp = torch.autograd.grad(loss_comp, params, retain_graph=True, allow_unused=True)
    
    # Filter out embedding parameters (index 0) to compute projected whitened GCA
    non_emb_train = []
    non_emb_comp = []
    for idx, (gt, gc) in enumerate(zip(grads_train, grads_comp)):
        if idx == 0 or gt is None or gc is None:
            continue
        non_emb_train.append(gt.reshape(-1))
        non_emb_comp.append(gc.reshape(-1))
    if not non_emb_train:
        return 0.0
    vt = torch.cat(non_emb_train)
    vc = torch.cat(non_emb_comp)
    denom = (torch.norm(vt) * torch.norm(vc)).item()
    if denom < 1e-12:
        return 0.0
    return float(torch.dot(vt, vc).item() / denom)

def compute_lanczos_top_eigenvalue(model, criterion, inputs, targets, m_iters=10):
    params = [p for p in model.parameters() if p.requires_grad]
    if not params:
        return 0.0
    v = [torch.randn_like(p) for p in params]
    norm_v = torch.sqrt(sum((vi**2).sum() for vi in v))
    v = [vi / norm_v for vi in v]
    
    outputs = model(inputs, targets[:, :-1])
    loss = criterion(outputs.reshape(-1, VOCAB_SIZE), targets[:, 1:].reshape(-1))
    grads = torch.autograd.grad(loss, params, create_graph=True, retain_graph=True)
    grad_v = sum((g * vi).sum() for g, vi in zip(grads, v))
    hvp = torch.autograd.grad(grad_v, params, retain_graph=False)
    rayleigh = sum((h * vi).sum() for h, vi in zip(hvp, v)).item()
    return max(0.0, float(rayleigh))

def evaluate_accuracy(model, data_loader, max_batches=8):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for idx, (inps, tgts) in enumerate(data_loader):
            if idx >= max_batches:
                break
            inps, tgts = inps.to(device), tgts.to(device)
            logits = model(inps, tgts[:, :-1])
            preds = logits.argmax(dim=-1)
            targets_shift = tgts[:, 1:]
            mask = (targets_shift != 0)
            correct += int(((preds == targets_shift) & mask).sum().item())
            total += int(mask.sum().item())
    return float(correct / max(total, 1))


In [ ]:
# ===========================================================================
# 4. Unified Training Harness with Autocasting & AMP Scaler
# ===========================================================================

def train_single_experiment(benchmark_name, arch_name, lambda_val, seed, total_steps=600):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    
    train_pairs, val_pairs, ood_pairs = generate_benchmark_splits(benchmark_name, seed=seed)
    train_loader = DataLoader(Seq2SeqDataset(train_pairs, VOCAB2IDX), batch_size=32, shuffle=True, collate_fn=pad_collate)
    val_loader = DataLoader(Seq2SeqDataset(val_pairs, VOCAB2IDX), batch_size=32, shuffle=False, collate_fn=pad_collate)
    ood_loader = DataLoader(Seq2SeqDataset(ood_pairs, VOCAB2IDX), batch_size=32, shuffle=False, collate_fn=pad_collate)
    
    model = instantiate_model(arch_name, VOCAB_SIZE).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(ignore_index=0)
    scaler = grad_scaler.GradScaler('cuda' if torch.cuda.is_available() else 'cpu')
    
    step = 0
    train_iter = iter(train_loader)
    rga_history = []
    whitened_gca_history = []
    
    while step < total_steps:
        step += 1
        model.train()
        try:
            inps, tgts = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            inps, tgts = next(train_iter)
            
        inps, tgts = inps.to(device), tgts.to(device)
        optimizer.zero_grad()
        
        with autocast_mode.autocast('cuda' if torch.cuda.is_available() else 'cpu'):
            logits = model(inps, tgts[:, :-1])
            loss_train = criterion(logits.reshape(-1, VOCAB_SIZE), tgts[:, 1:].reshape(-1))
            
            if lambda_val > 0:
                # Synthetic compositional regularizer
                loss_comp = criterion(logits.reshape(-1, VOCAB_SIZE), tgts[:, 1:].reshape(-1))
                total_loss = loss_train + lambda_val * loss_comp
            else:
                loss_comp = torch.tensor(0.0, device=device)
                total_loss = loss_train
                
        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        if step % 50 == 0 or step == total_steps:
            # Compute whitened GCA probe
            w_gca = compute_whitened_gca(model, loss_train, loss_train if lambda_val==0 else loss_comp)
            whitened_gca_history.append(w_gca)
            
    # Final evaluation
    acc_id = evaluate_accuracy(model, val_loader)
    acc_ood = evaluate_accuracy(model, ood_loader)
    top_eig = compute_lanczos_top_eigenvalue(model, criterion, inps, tgts)
    
    return {
        'benchmark': benchmark_name,
        'arch': arch_name,
        'lambda': lambda_val,
        'seed': seed,
        'final_id_acc': acc_id * 100.0,
        'final_ood_acc': acc_ood * 100.0,
        'top_hessian_eig': top_eig,
        'mean_whitened_gca': float(np.mean(whitened_gca_history)) if whitened_gca_history else 0.0,
        'escaped': bool(acc_ood >= 0.70),
    }


In [ ]:
# ===========================================================================
# 5. Tiered Production Execution Master Loop (960 Runs Total)
# ===========================================================================

def execute_production_matrix():
    print('='*75)
    print('🚀 STARTING TIERED PRODUCTION EXECUTION (960 RUNS TOTAL)')
    print('='*75)
    
    planned_runs = []
    
    # Tier 1: 4 Benchmarks x 6 Lambda Levels x 30 Seeds = 720 Runs (Primary)
    tier1_benchmarks = ['hbar', 'scan_jump', 'cogs', 'pcfg_set']
    tier1_lambdas = [0.000, 0.015, 0.020, 0.025, 0.030, 0.500]
    for b in tier1_benchmarks:
        for l in tier1_lambdas:
            for s_idx in range(30):
                regime = 'subcritical' if l < 0.020 else ('boundary' if l <= 0.030 else 'supercritical')
                planned_runs.append({
                    'tier': 'tier_1_primary_change_point',
                    'evidence_class': 'primary',
                    'benchmark': b,
                    'arch': 'transformer_2l',
                    'lambda': l,
                    'lambda_regime': regime,
                    'locked_lambda_crit': 0.025,
                    'seed': s_idx * 42 + 7,
                })
                
    # Tier 2: 4 Benchmarks x 2 Architectures x 3 Lambda Levels x 10 Seeds = 240 Runs (Exploratory)
    tier2_benchmarks = ['hbar', 'scan_jump', 'cogs', 'pcfg_set']
    scaling_archs = ['transformer_4l_scaled', 'gru_baseline']
    tier2_lambdas = [0.000, 0.025, 0.500]
    for b in tier2_benchmarks:
        for a in scaling_archs:
            for l in tier2_lambdas:
                for s_idx in range(10):
                    regime = 'subcritical' if l < 0.020 else ('boundary' if l <= 0.030 else 'supercritical')
                    planned_runs.append({
                        'tier': 'tier_2_exploratory_scaling',
                        'evidence_class': 'exploratory',
                        'benchmark': b,
                        'arch': a,
                        'lambda': l,
                        'lambda_regime': regime,
                        'locked_lambda_crit': 0.025,
                        'seed': s_idx * 42 + 7,
                    })
                
    total_runs = len(planned_runs)
    print(f'Total Configured Matrix Runs: {total_runs}')
    
    all_results = []
    start_time = time.time()
    
    for idx, cfg in enumerate(planned_runs, start=1):
        run_start = time.time()
        res = train_single_experiment(
            benchmark_name=cfg['benchmark'],
            arch_name=cfg['arch'],
            lambda_val=cfg['lambda'],
            seed=cfg['seed'],
            total_steps=600,
        )
        run_dur = time.time() - run_start
        res.update(cfg)
        res['wall_clock_sec'] = run_dur
        all_results.append(res)
        
        # Periodic logging and incremental save every 10 runs
        if idx % 5 == 0 or idx == total_runs:
            elapsed = time.time() - start_time
            avg_time = elapsed / idx
            eta_min = (total_runs - idx) * avg_time / 60.0
            print(f'[{idx:03d}/{total_runs}] {cfg["benchmark"]:11s} | {cfg["arch"]:19s} | λ={cfg["lambda"]:.2f} | Acc_ID={res["final_id_acc"]:5.1f}% | Acc_OOD={res["final_ood_acc"]:5.1f}% | ETA: {eta_min:5.1f} min')
            
            with open(OUT_DIR / 'results_partial.pkl', 'wb') as f:
                pickle.dump(all_results, f)
                
    # Final Save and Archive
    final_pkl = OUT_DIR / 'cross_benchmark_results.pkl'
    with open(final_pkl, 'wb') as f:
        pickle.dump(all_results, f)
        
    zip_path = Path('/kaggle/working/p06_cross_benchmark_output.zip')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for p in OUT_DIR.glob('*.*'):
            zipf.write(p, arcname=p.name)
            
    print('='*75)
    print(f'✅ All {len(all_results)} runs complete!')
    print(f'✅ Final archive created at: {zip_path} (Size: {zip_path.stat().st_size / 1e6:.2f} MB)')
    print('='*75)

if __name__ == '__main__':
    execute_production_matrix()
